# Model Router and LLM Integration

This notebook defines how the chatbot decides which diagnostic helper model to call and how to wrap the result for an LLM response.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.diagnostic_models import predict_for_backend
from src.router import route_request_llm, route_request_fallback


In [ ]:
# Recommended LLM setup
llm_recommendations = {
    'best_cloud_quality': 'GPT-4o or Gemini 2.5 Pro for Egyptian Arabic medical explanations and safety-aware follow-up',
    'open_source_arabic_capable': 'Qwen2.5-7B-Instruct or Llama-3.1-8B-Instruct via Hugging Face/Transformers',
    'medical_rag': 'Add retrieval from approved Arabic medical content before final answers',
    'guardrails': ['no final diagnosis', 'urgent symptom escalation', 'ask for missing context']
}
print(json.dumps(llm_recommendations, indent=2))


In [ ]:
# Prompt template that the backend sends to the LLM after model prediction.
def build_llm_prompt(user_message: str, model_output: dict) -> str:
    return f'''
أنت طبيب مصري ودود وخبير. اشرح النتيجة الطبية التالية للمريض بالعامية المصرية المبسطة جداً (اللهجة المصرية).
لا تشخص المرض كتشخيص نهائي، بل وضح له المعنى ببساطة.
وضح أي علامات خطر حمراء (Red Flags) بشكل صريح جداً إذا وجدتها.
اسأله أسئلة متابعة فقط إذا كانت ستغير من درجة خطورة الحالة.
الرسالة من المستخدم: {user_message}
مخرجات النموذج الطبي (JSON): {json.dumps(model_output, ensure_ascii=False)}
المطلوب إرجاعه بالتفصيل:
1. ملخص بسيط للحالة بالعامية المصرية.
2. معنى المصطلحات غير الطبيعية (مثل فقر الدم، وظائف الكلى، إلخ) بأسلوب مصري مبسط.
3. الخطوة التالية اللي المفروض يعملها (زي حجز ميعاد مع دكتور أو الذهاب للطوارئ).
4. أسئلة المتابعة لو فيه حاجة ناقصة.
'''.strip()


In [ ]:
# End-to-end example
message = 'CBC result: hemoglobin low and WBC high, should I worry?'
payload = {
    'age': 34, 'sex': 'female', 'hemoglobin_g_dl': 10.2, 'wbc_10e3_ul': 13.4,
    'platelets_10e3_ul': 210, 'rbc_10e6_ul': 4.1, 'mcv_fl': 74, 'mch_pg': 25,
    'neutrophils_pct': 82, 'lymphocytes_pct': 12
}
model_name = route_request_llm(message, payload)
model_output = predict_for_backend(model_name, payload, ROOT)
print('Routed model:', model_name)
print(json.dumps(model_output, indent=2, ensure_ascii=False))
print('\nLLM PROMPT:\n', build_llm_prompt(message, model_output))
